In [1]:
import pandas as pd
from sklearn.metrics import precision_recall_curve,auc,accuracy_score,precision_score,recall_score, f1_score, average_precision_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE


In [2]:
#mounting drive in google collab
from google.colab import drive
drive.mount('/content/drive', force_remount = True)


Mounted at /content/drive


In [3]:

df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/creditcard.csv')
df.dropna(inplace = True)

In [4]:
# @title Using train_test_split

X=df.drop(columns=['Class','Time']) #removing Time column
y=df['Class']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.35,random_state=43,shuffle=True)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))



Training samples: 185124
Testing samples: 99683


In [6]:
# @title GridSearchCV
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_leaf": [100],
    "max_features": ["sqrt"]
}

rf = RandomForestClassifier(
    random_state=42,
    n_jobs=1,
    class_weight="balanced",
    min_samples_split=100,
    max_samples=0.75
)

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring="recall",
    cv=3,
    n_jobs=-1,
    verbose=2,
    refit=False
)

# Tune on a stratified subset for speed
X_tune, _, y_tune, _ = train_test_split(
    X_train,
    y_train,
    train_size=min(80000, len(X_train)),
    stratify=y_train,
    random_state=42
)

grid_search.fit(X_tune, y_tune)

print("Best parameters:", grid_search.best_params_)
print("Best CV average precision:", grid_search.best_score_)


Fitting 3 folds for each of 6 candidates, totalling 18 fits
Best parameters: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 100, 'n_estimators': 100}
Best CV average precision: 0.837037037037037


In [7]:
grid_results = pd.DataFrame(grid_search.cv_results_)

print(grid_results[
    [
        "params",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].sort_values("rank_test_score").to_string(index=False))

                                                                                   params  mean_test_score  std_test_score  rank_test_score
{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 100, 'n_estimators': 100}         0.837037        0.027716                1
   {'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 100, 'n_estimators': 100}         0.837037        0.027716                1
   {'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 100, 'n_estimators': 200}         0.837037        0.027716                1
  {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 100, 'n_estimators': 100}         0.837037        0.027716                1
{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 100, 'n_estimators': 200}         0.829630        0.020951                5
  {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 100, 'n_estimators': 200}         0.829630        0.020951                5


In [8]:
# @title RandomSearchCV

from scipy.stats import randint

param_dist = {
    "n_estimators": [80, 100, 150],
    "max_depth": [5, 10, None],
    "min_samples_split": [50, 100],
    "max_features": ["sqrt"],
    "max_samples": [0.5,0.75]
}

rf_random = RandomForestClassifier(
    random_state=42,
    n_jobs=1,
    class_weight="balanced_subsample"
)

random_search = RandomizedSearchCV(
    estimator=rf_random,
    param_distributions=param_dist,
    n_iter=5,                 # Only 5 sampled combinations
    scoring="recall",
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=2,
    refit=False
)

random_search.fit(X_tune, y_tune)

print("Best parameters:", random_search.best_params_)
print("Best CV average precision:", random_search.best_score_)

Fitting 3 folds for each of 5 candidates, totalling 15 fits
Best parameters: {'n_estimators': 100, 'min_samples_split': 100, 'max_samples': 0.5, 'max_features': 'sqrt', 'max_depth': 10}
Best CV average precision: 0.8148148148148149


In [9]:
random_results = pd.DataFrame(random_search.cv_results_)

print(random_results[
    [
        "params",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].sort_values("rank_test_score").to_string(index=False))

                                                                                                         params  mean_test_score  std_test_score  rank_test_score
   {'n_estimators': 100, 'min_samples_split': 100, 'max_samples': 0.5, 'max_features': 'sqrt', 'max_depth': 10}         0.814815        0.027716                1
{'n_estimators': 150, 'min_samples_split': 100, 'max_samples': 0.75, 'max_features': 'sqrt', 'max_depth': None}         0.807407        0.027716                2
  {'n_estimators': 150, 'min_samples_split': 50, 'max_samples': 0.5, 'max_features': 'sqrt', 'max_depth': None}         0.770370        0.037771                3
    {'n_estimators': 100, 'min_samples_split': 50, 'max_samples': 0.5, 'max_features': 'sqrt', 'max_depth': 10}         0.770370        0.020951                3
  {'n_estimators': 80, 'min_samples_split': 50, 'max_samples': 0.75, 'max_features': 'sqrt', 'max_depth': None}         0.755556        0.036289                5


In [24]:
print("\nModel Evaluation")
print("-------------------------")
print("Accuracy:",
      accuracy_score(y_test, y_pred))

print("Precision:",
      precision_score(y_test, y_pred, zero_division=0))

print("Recall:",
      recall_score(y_test, y_pred, zero_division=0))

print("F1 Score:",
      f1_score(y_test, y_pred, zero_division=0))

# print("Average Precision (PR-AUC summary):",
#       average_precision_score(y_test, y_prob))


Model Evaluation
-------------------------
Accuracy: 0.9994984099595718
Precision: 0.9387755102040817
Recall: 0.770949720670391
F1 Score: 0.8466257668711656
